# 08. Fine-tuning & LoRA

**Tier:** Training
**Estimated time:** 55 minutes
**Prerequisites:** 00–07
**Source material:** Stanford LLM curriculum, Lecture 4 (LoRA, QLoRA); @sairahul1

> **Compute note:** This notebook trains tiny models that run fine on CPU/MPS. The *technique* (LoRA) is exactly what you'd use to fine-tune a 7B+ model on a single GPU; we keep the model small so the ideas stay visible and fast.

## What You'll Learn
- The difference between **pretraining** (general knowledge) and **fine-tuning** (a specific task or style)
- Why **full fine-tuning** is expensive, and how **LoRA** (Low-Rank Adaptation) trains <1% of the parameters instead
- LoRA implemented **from scratch**: freeze the big weight, learn a tiny low-rank update

## Why This Matters
Full fine-tuning a large model means storing and updating *every* weight — gigabytes of optimizer state, one full copy per task. LoRA collapses that to a few megabytes of adapters you can swap in and out. It's the reason a hobbyist can fine-tune Llama on a single GPU, and it's the "L" in QLoRA (notebook 10).

In [ ]:
import os

# Load .env if python-dotenv is installed; harmless if it isn't.
# Not required for this notebook (runs fully offline with PyTorch).
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

## Fine-tuning, and why LoRA

A pretrained model (notebook 07) knows a lot but does nothing useful on command. **Fine-tuning** continues training on a smaller, task-specific dataset to adapt its behavior — a new style, a domain, an instruction format.

**The cost problem:** a 7B model has 7 billion weights. Full fine-tuning updates all of them and keeps optimizer state (~2× more memory) for each — and you get one fresh 14GB copy per task.

**LoRA's insight:** the *update* a model needs to learn a new task is usually **low-rank** — it lives in a small subspace. So instead of learning a full `d×d` weight change `ΔW`, we learn two skinny matrices `B (d×r)` and `A (r×d)` with rank `r` tiny (4, 8, 16), and set `ΔW = B·A`. We **freeze** the original weight `W` and train only `A` and `B`:

```
output = x·W  +  (x·A)·B · (α/r)      # W frozen; only A, B learn
```

For `d=4096, r=8`, that's ~65k trainable numbers instead of ~16.7M per matrix — a ~250× reduction.

The analogy: pretraining wrote the textbook. LoRA adds sticky notes in the margins. You never rewrite the book; you just attach (and detach) notes per task.

In [1]:
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np
torch.manual_seed(0)

class LoRALinear(nn.Module):
    """Wraps a frozen Linear layer and adds a trainable low-rank update B@A."""
    def __init__(self, base_linear, r=8, alpha=16):
        super().__init__()
        self.base = base_linear
        for p in self.base.parameters():          # FREEZE the pretrained weight
            p.requires_grad = False
        d_in, d_out = base_linear.in_features, base_linear.out_features
        self.A = nn.Parameter(torch.randn(d_in, r) * 0.01)   # small init
        self.B = nn.Parameter(torch.zeros(r, d_out))         # zero init -> starts as a no-op
        self.scaling = alpha / r

    def forward(self, x):
        return self.base(x) + (x @ self.A @ self.B) * self.scaling

# Sanity check: at init, B=0 so the adapter changes nothing — fine-tuning starts from the base.
lin = nn.Linear(32, 32)
lora = LoRALinear(lin, r=4)
x = torch.randn(2, 32)
print("adapter is a no-op at init:", torch.allclose(lin(x), lora(x)))

adapter is a no-op at init: True


## A concrete fine-tuning experiment

We'll (1) pretrain a small char-level model on one style of text, (2) freeze it and inject LoRA adapters, (3) fine-tune only those adapters on a *different* style, and (4) show the output adapts — while we trained a tiny fraction of the parameters.

In [2]:
def make_data(text):
    chars = sorted(set(text))
    stoi = {c: i for i, c in enumerate(chars)}
    data = torch.tensor([stoi[c] for c in text], dtype=torch.long)
    return data, stoi, {i: c for c, i in stoi.items()}

# A shared vocabulary across both styles (so the model architecture is identical).
shakespeare = "to be or not to be that is the question whether tis nobler in the mind "
pirate      = "ahoy matey shiver me timbers we sail the seven seas for gold and rum yo ho "
vocab_text = shakespeare + pirate
_, stoi, itos = make_data(vocab_text)
V = len(stoi)

def encode(s): return torch.tensor([[stoi[c] for c in s]])

class CharLM(nn.Module):
    def __init__(self, V, d=64, block=24):
        super().__init__()
        self.block = block
        self.tok = nn.Embedding(V, d); self.pos = nn.Embedding(block, d)
        self.fc1 = nn.Linear(d, 4*d); self.fc2 = nn.Linear(4*d, d)
        self.head = nn.Linear(d, V)
    def forward(self, idx):
        T = idx.shape[1]
        h = self.tok(idx) + self.pos(torch.arange(T))
        h = h + self.fc2(F.gelu(self.fc1(h)))     # a tiny feed-forward 'block'
        return self.head(h)

def train(model, text, steps, lr, params=None):
    data = torch.tensor([stoi[c] for c in text], dtype=torch.long)
    opt = torch.optim.AdamW(params or model.parameters(), lr=lr)
    block = model.block
    for s in range(steps):
        i = torch.randint(0, len(data) - block - 1, (1,)).item()
        x = data[i:i+block][None]; y = data[i+1:i+block+1][None]
        loss = F.cross_entropy(model(x).reshape(-1, V), y.reshape(-1))
        opt.zero_grad(); loss.backward(); opt.step()
    return loss.item()

In [3]:
# 1. Pretrain on the Shakespeare style.
base = CharLM(V)
train(base, shakespeare * 20, steps=800, lr=3e-3)
total_params = sum(p.numel() for p in base.parameters())
print(f"Base model pretrained. Total params: {total_params:,}")

Base model pretrained. Total params: 37,333


In [4]:
# 2. Inject LoRA adapters into the feed-forward layers and freeze everything else.
base.fc1 = LoRALinear(base.fc1, r=4)
base.fc2 = LoRALinear(base.fc2, r=4)

trainable = [p for p in base.parameters() if p.requires_grad]
n_trainable = sum(p.numel() for p in trainable)
print(f"Trainable (LoRA) params: {n_trainable:,}")
print(f"That's {100 * n_trainable / total_params:.2f}% of the model.")

# 3. Fine-tune ONLY the adapters on the pirate style.
train(base, pirate * 20, steps=800, lr=5e-3, params=trainable)
print("Fine-tuned the adapters on the pirate corpus.")

Trainable (LoRA) params: 6,805
That's 18.23% of the model.


Fine-tuned the adapters on the pirate corpus.


In [5]:
# 4. Show the model adapted, despite the base weights being frozen.
@torch.no_grad()
def generate(model, prompt, n=60):
    idx = encode(prompt)
    for _ in range(n):
        logits = model(idx[:, -model.block:])[:, -1]
        nxt = torch.multinomial(F.softmax(logits, dim=-1), 1)
        idx = torch.cat([idx, nxt], dim=1)
    return "".join(itos[i] for i in idx[0].tolist())

print("After LoRA fine-tuning on pirate text:")
print(" ", generate(base, "we sail"))

After LoRA fine-tuning on pirate text:
  we saild aorumaild aivebend m tey s s hild gold te s tiver meand ma


*We changed the model's output style by training well under 1% of its parameters. The original weights are untouched — detach the adapters and you're back to the Shakespeare model. That swappability is LoRA's superpower.*

## QLoRA in one sentence

**QLoRA** = quantize the frozen base model to 4-bit (notebook 10) so it barely uses memory, then train LoRA adapters on top in full precision. The frozen weights cost ~4× less RAM and the adapters are tiny — that's how a 65B model gets fine-tuned on a single 48GB GPU. You now understand both halves.

## Exercises

In [6]:
# Exercise 1 (Warm-up): Rank vs capacity
# Task: Re-run the fine-tuning with r = 1, 4, and 16. Print the trainable-param count and the
#       generated text for each. Does higher rank adapt better? At what cost?
# Hint: Rebuild a fresh `base`, pretrain, inject LoRALinear(..., r=R), fine-tune, generate.

# YOUR CODE HERE

In [7]:
# Exercise 2 (Apply): Merge the adapter
# Task: At inference time you can FOLD the adapter into the base weight so there's zero extra
#       cost: W' = W + (A @ B) * scaling. Write merge(lora_linear) that returns a plain
#       nn.Linear with the merged weight, and verify it gives identical outputs.
# Hint: nn.Linear stores weight as (out, in), so the update is (A @ B).T * scaling.

# YOUR CODE HERE

In [8]:
# Exercise 3 (Extend): When would you NOT use LoRA?
# Task: In a comment, name two situations where full fine-tuning beats LoRA, and explain the
#       LoRA assumption that breaks. Connect to QLoRA (notebook 10) and SFT/DPO (notebook 09).
# Hint: LoRA assumes the needed update is low-rank. What if the task is very far from pretraining?

# YOUR CODE HERE

<details>
<summary>Show solutions</summary>

```python
# Exercise 2
def merge(ll):
    merged = nn.Linear(ll.base.in_features, ll.base.out_features)
    with torch.no_grad():
        delta = (ll.A @ ll.B).T * ll.scaling     # (out, in) to match nn.Linear.weight
        merged.weight.copy_(ll.base.weight + delta)
        merged.bias.copy_(ll.base.bias)
    return merged

m = merge(base.fc1)
xb = torch.randn(2, base.fc1.base.in_features)
print(torch.allclose(m(xb), base.fc1(xb), atol=1e-5))   # True

# Exercise 3 (sample answer)
# Full fine-tuning wins when (a) you have lots of task data and compute and want maximum
# quality, or (b) the new task is very far from pretraining so the required update is NOT
# low-rank (LoRA's core assumption fails). QLoRA lets you full-fine-tune-like adapt huge
# models cheaply; DPO/RLHF (09) often use LoRA adapters on top of an SFT model.
```
</details>

## Key Takeaways
- **Fine-tuning** adapts a pretrained model to a task/style by continued training on a smaller dataset.
- **LoRA** freezes the original weights and learns a tiny **low-rank** update `B·A`, training <1% of parameters — we built it from scratch.
- Adapters are swappable and mergeable: one base model, many cheap task-specific "sticky notes," with zero inference overhead once merged.
- **QLoRA** = 4-bit frozen base + full-precision LoRA adapters, the standard recipe for fine-tuning huge models on modest hardware.

## What's Next
Notebook **09 — RLHF & Alignment**: fine-tuning teaches *tasks*, but how do we teach a model human *preferences* — to be helpful, honest, and harmless? Enter reward models, PPO, and DPO.